# Checkpoint — KMeans & DBSCAN

Dit notebook wordt automatisch gecheckt via pytest (GitHub Actions). De checks voeren je notebook uit en valideren of bepaalde variabelen bestaan en binnen de verwachte ranges vallen. **Je krijgt geen hints in de foutmeldingen: het is aan jou om correct werkende code te schrijven.**

## BELANGRIJK: gebruik exact de genoemde variabelennamen

De checks zoeken volgende variabelen in je notebook. Andere namen worden niet gezien:

- `best_k_wine` — de gekozen waarde van k voor de Wine-dataset (int)
- `silhouette_wine` — de silhouette score van die keuze (float)
- `image` — de ingeladen afbeelding (3D numpy-array)
- `n_colors` — het tuple van aantal kleuren `(10, 8, 6, 4, 2)`
- `segmented_imgs` — de lijst van gesegmenteerde afbeeldingen (5 elementen, zelfde vorm als `image`)
- `best_k_iris` — de gekozen waarde van k voor de Iris-dataset (int)
- `silhouette_iris` — de silhouette score van die keuze (float)
- `best_k_mall` — de gekozen waarde van k voor de Mall-dataset (int)
- `silhouette_mall` — de silhouette score van die keuze (float)
- `antwoord_iris` — meerkeuzevraag 1: `"A"`, `"B"`, `"C"` of `"D"`
- `antwoord_dbscan` — meerkeuzevraag 2: `"A"`, `"B"`, `"C"` of `"D"`

Gebruik overal `random_state=42` (en `n_init=10`) bij KMeans zodat je resultaten reproduceerbaar zijn en binnen de verwachte drempels vallen. Werk met de lokale afbeelding `images/butterfly.jpg` (er wordt niets gedownload).

## Oefening 1 — Wijn

Cluster de Wine-dataset zo optimaal mogelijk met KMeans. De dataset heeft 13 features op verschillende schalen, dus **schaal eerst met `StandardScaler`**.

Probeer verschillende waarden van `k`, bereken telkens de **silhouette score** en kies de `k` met de hoogste score.

Bewaar je keuze in `best_k_wine` en de bijhorende score in `silhouette_wine`.

In [ ]:
import numpy as np
from sklearn import datasets
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

wine = datasets.load_wine()
data = wine.data

# Schaal de data zodat alle features gelijk gewicht krijgen
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

In [ ]:
# TODO: bereken voor elke k in range(2, 8) de silhouette score (KMeans met
# random_state=42, n_init=10). Bewaar de k met de hoogste score in best_k_wine
# en die score in silhouette_wine.
best_k_wine = 2
silhouette_wine = 0.0

for k in range(2, 8):
    # 1. Maak het KMeans model aan
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    
    # 2. Fit het model en voorspel de cluster labels voor de geschaalde data
    cluster_labels = kmeans.fit_predict(data_scaled)
    
    # 3. Bereken de silhouette score
    score = silhouette_score(data_scaled, cluster_labels)
    
    # 4. Kijk of dit de beste score tot nu toe is
    if score > silhouette_wine:
        silhouette_wine = score
        best_k_wine = k

print(f"De beste k is: {best_k_wine}")
print(f"De bijbehorende hoogste silhouette score is: {silhouette_wine}")

## Oefening 2 — Image Segmentation

We gebruiken clustering om een afbeelding in stukken met gelijkaardige kleuren te verdelen. Gebruik de lokale afbeelding `images/butterfly.jpg` (533×800 pixels, 3 kleurkanalen).

Hervorm de 3D-array tot een `(n_pixels, 3)` matrix met `reshape(-1, 3)`, train dan KMeans met `n_clusters` in `(10, 8, 6, 4, 2)` en vervang elke pixel door de centroidkleur van zijn cluster. Hervorm het resultaat terug naar de originele vorm.

Bewaar het tuple `n_colors = (10, 8, 6, 4, 2)` en de 5 gesegmenteerde afbeeldingen in `segmented_imgs`.

In [1]:
import numpy as np
from PIL import Image
from sklearn.cluster import KMeans

# Gebruik het lokale voorbeeldbeeld (geen download nodig)
image = np.asarray(Image.open('images/butterfly.jpg'))
print('image shape:', image.shape)

# Hervorm de 3D-array (pixels, breedte, kanalen) naar een (n_pixels, 3) matrix
X = image.reshape(-1, 3)

image shape: (533, 800, 3)


In [2]:
# TODO: train voor elk aantal kleuren een KMeans (random_state=42, n_init=10)
# op X, vervang elke pixel door de centroidkleur van zijn cluster en hervorm
# terug naar image.shape. Bewaar alles in segmented_imgs (volgorde van n_colors).
n_colors = (10, 8, 6, 4, 2)
segmented_imgs = []

for n_clusters in n_colors:
    # 1. Maak en train het KMeans model
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    kmeans.fit(X)
    
    # 2. Haal de centroids (de nieuwe 'hoofdkleuren') op
    # kmeans.cluster_centers_ bevat de RGB waarden van de gevonden centroids
    centroids = kmeans.cluster_centers_
    
    # 3. Haal de labels op (aan welk cluster behoort elke originele pixel?)
    # kmeans.labels_ is een lijst die voor elke pixel aangeeft bij welk centroid hij hoort (0, 1, 2, ...)
    labels = kmeans.labels_
    
    # 4. Vervang de originele pixels door de centroid kleuren
    # Dit is een handige NumPy truc: we gebruiken de array met labels als indexen
    # om de juiste centroid kleur voor elke pixel op te halen.
    segmented_img_pixels = centroids[labels]
    
    # De kleuren moeten vaak integers (gehele getallen) zijn voor afbeeldingen (0-255)
    segmented_img_pixels = segmented_img_pixels.astype(np.uint8)
    
    # 5. Hervorm de 1D lijst weer naar de originele 3D beeldvorm
    segmented_img = segmented_img_pixels.reshape(image.shape)
    
    # Voeg toe aan de lijst
    segmented_imgs.append(segmented_img)

## Oefening 3 — KMeans vs DBSCAN op Iris

De Iris-dataset bevat 3 soorten die in de feature-ruimte goed gescheiden, **bolvormige** clusters van **gelijkaardige dichtheid** vormen.

Pas zowel KMeans (verschillende `k`) als DBSCAN (verschillende `eps`/`min_samples`) toe op de **gestandaardiseerde** data en vergelijk met de silhouette score. Bewaar je gekozen `k` in `best_k_iris` en de hoogste silhouette in `silhouette_iris`. Beantwoord daarna vraag 1 in `antwoord_iris`.

In [3]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

iris = load_iris()
data_scaled = StandardScaler().fit_transform(iris.data)

best_k_iris = 2
silhouette_iris = 0.0

# 1. KMeans evalueren voor k=2 tot k=5
print("--- KMeans Scores ---")
for k in range(2, 6):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_kmeans = kmeans.fit_predict(data_scaled)
    score = silhouette_score(data_scaled, labels_kmeans)
    print(f"KMeans met k={k}: Silhouette Score = {score:.4f}")
    
    if score > silhouette_iris:
        silhouette_iris = score
        best_k_iris = k

# 2. DBSCAN evalueren met verschillende combinaties
print("\n--- DBSCAN Scores ---")
eps_values = [0.3, 0.5, 0.7, 0.9]
min_samples_values = [3, 4, 5]

for eps in eps_values:
    for min_samples in min_samples_values:
        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
        labels_dbscan = dbscan.fit_predict(data_scaled)
        
        # Silhouette score vereist minimaal 2 clusters (en niet alleen maar noise)
        unieke_labels = set(labels_dbscan)
        if len(unieke_labels) > 1:
            score = silhouette_score(data_scaled, labels_dbscan)
            print(f"DBSCAN met eps={eps}, min_samples={min_samples}: Silhouette Score = {score:.4f}")
        else:
            print(f"DBSCAN met eps={eps}, min_samples={min_samples}: Geen geldige clusters gevonden")

print(f"\nUiteindelijke keuze KMeans:")
print(f"best_k_iris = {best_k_iris}")
print(f"silhouette_iris = {silhouette_iris:.4f}")

--- KMeans Scores ---
KMeans met k=2: Silhouette Score = 0.5818
KMeans met k=3: Silhouette Score = 0.4599
KMeans met k=4: Silhouette Score = 0.3869
KMeans met k=5: Silhouette Score = 0.3459

--- DBSCAN Scores ---
DBSCAN met eps=0.3, min_samples=3: Silhouette Score = -0.1473
DBSCAN met eps=0.3, min_samples=4: Silhouette Score = -0.2073
DBSCAN met eps=0.3, min_samples=5: Silhouette Score = -0.1942
DBSCAN met eps=0.5, min_samples=3: Silhouette Score = 0.1597
DBSCAN met eps=0.5, min_samples=4: Silhouette Score = 0.3646
DBSCAN met eps=0.5, min_samples=5: Silhouette Score = 0.3565
DBSCAN met eps=0.7, min_samples=3: Silhouette Score = 0.5081
DBSCAN met eps=0.7, min_samples=4: Silhouette Score = 0.5081
DBSCAN met eps=0.7, min_samples=5: Silhouette Score = 0.5234
DBSCAN met eps=0.9, min_samples=3: Silhouette Score = 0.5217
DBSCAN met eps=0.9, min_samples=4: Silhouette Score = 0.5217
DBSCAN met eps=0.9, min_samples=5: Silhouette Score = 0.5217

Uiteindelijke keuze KMeans:
best_k_iris = 2
silhoue

## Oefening 4 — KMeans vs DBSCAN op Mall

De Mall-dataset (`data/Mall_Customers.csv`) bevat klanten met `Age`, `Annual Income (k$)` en `Spending Score (1-100)`. De clusters hebben **sterk verschillende dichtheden**, waardoor DBSCAN het vaak moeilijk heeft.

Pas KMeans (verschillende `k`) en DBSCAN (verschillende `eps`/`min_samples`) toe op de **gestandaardiseerde** data en vergelijk met de silhouette score. Bewaar je gekozen `k` in `best_k_mall` en de hoogste silhouette in `silhouette_mall`. Beantwoord daarna vraag 2 in `antwoord_dbscan`.

In [4]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

mall = pd.read_csv('data/Mall_Customers.csv')
features = mall[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']]
data_scaled = StandardScaler().fit_transform(features)

best_k_mall = 2
silhouette_mall = 0.0

print("--- KMeans Resultaten ---")
for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_kmeans = kmeans.fit_predict(data_scaled)
    score = silhouette_score(data_scaled, labels_kmeans)
    print(f"KMeans (k={k}): Silhouette = {score:.4f}")
    
    if score > silhouette_mall:
        silhouette_mall = score
        best_k_mall = k

print("\n--- DBSCAN Resultaten ---")
eps_values = [0.3, 0.5, 0.7, 0.9, 1.2]
min_samples_values = [3, 4, 5, 7]
best_dbscan_score = -1

for eps in eps_values:
    for min_samples in min_samples_values:
        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
        labels_dbscan = dbscan.fit_predict(data_scaled)
        
        # Silhouette score vereist minimaal 2 clusters (en niet alleen noise)
        unieke_labels = set(labels_dbscan)
        if len(unieke_labels) > 1 and len(unieke_labels) < len(data_scaled):
            score = silhouette_score(data_scaled, labels_dbscan)
            if score > best_dbscan_score:
                best_dbscan_score = score
            print(f"DBSCAN (eps={eps}, min_samples={min_samples}): Silhouette = {score:.4f}")

print(f"\nDe hoogste KMeans score is {silhouette_mall:.4f} bij k={best_k_mall}.")
print(f"De hoogste DBSCAN score is {best_dbscan_score:.4f}.")

--- KMeans Resultaten ---
KMeans (k=2): Silhouette = 0.3355
KMeans (k=3): Silhouette = 0.3578
KMeans (k=4): Silhouette = 0.4040
KMeans (k=5): Silhouette = 0.4166
KMeans (k=6): Silhouette = 0.4284
KMeans (k=7): Silhouette = 0.4172
KMeans (k=8): Silhouette = 0.4082
KMeans (k=9): Silhouette = 0.4177
KMeans (k=10): Silhouette = 0.4066

--- DBSCAN Resultaten ---
DBSCAN (eps=0.3, min_samples=3): Silhouette = -0.0796
DBSCAN (eps=0.3, min_samples=4): Silhouette = -0.0796
DBSCAN (eps=0.3, min_samples=5): Silhouette = -0.1822
DBSCAN (eps=0.3, min_samples=7): Silhouette = -0.1775
DBSCAN (eps=0.5, min_samples=3): Silhouette = 0.1316
DBSCAN (eps=0.5, min_samples=4): Silhouette = 0.1122
DBSCAN (eps=0.5, min_samples=5): Silhouette = 0.1845
DBSCAN (eps=0.5, min_samples=7): Silhouette = 0.2236
DBSCAN (eps=0.7, min_samples=3): Silhouette = 0.1393
DBSCAN (eps=0.7, min_samples=4): Silhouette = 0.1446
DBSCAN (eps=0.7, min_samples=5): Silhouette = 0.2429
DBSCAN (eps=0.7, min_samples=7): Silhouette = 0.1099


## Meerkeuzevragen

### Vraag 1 — Iris
Op de Iris-dataset haalt KMeans een veel hogere silhouette score dan DBSCAN. Wat is de belangrijkste reden?

- **A)** De clusters zijn bolvormig en hebben een gelijkaardige dichtheid, waarvoor KMeans goed geschikt is; DBSCAN kampt met de dichtheidsverschillen.
- **B)** KMeans werkt zonder features te standaardiseren.
- **C)** DBSCAN kan nooit meer dan 2 clusters vinden.
- **D)** KMeans kent het aantal clusters op voorhand, en dat is altijd beter.

Bewaar je antwoord (`"A"`, `"B"`, `"C"` of `"D"`) in `antwoord_iris`.

### Vraag 2 — DBSCAN
Wanneer is DBSCAN een betere keuze dan KMeans?

- **A)** Wanneer de clusters een onregelmatige, niet-bolvormige vorm hebben of er outliers/ruis zijn.
- **B)** Wanneer alle clusters bolvormig en even groot zijn.
- **C)** Wanneer je altijd precies `k` clusters nodig hebt.
- **D)** Wanneer de dataset klein is en je snel wilt clusteren.

Bewaar je antwoord (`"A"`, `"B"`, `"C"` of `"D"`) in `antwoord_dbscan`.

In [ ]:
# Vraag 1: antwoord_iris = "A", "B", "C" of "D"
antwoord_iris = "A"

# Vraag 2: antwoord_dbscan = "A", "B", "C" of "D"
antwoord_dbscan = "A"
